In [2]:
import requests
import pandas as pd
import numpy as np
import polars as pl

In [3]:
# Fetch help contacts data
help_contacts = requests.get("http://api.justgood.work/api/v2/help-contacts")
help_contacts_json = help_contacts.json()["data"]

In [4]:
# Convert help contacts json to a pandas Dataframe
hc_df = pd.DataFrame.from_records(help_contacts_json)

# view the first 2 records
hc_df.head()

,_id,deviceId,appId,locale,route,phoneNumber,organization,email,createdAt,updatedAt,__v,language
0,6463a77057f25744aff36429,jgw-vanilla-afe14226d1d2386e-TJK -> GBR,jgw-vanilla,TJK,TJK -> GBR,08000 121 700,,,2023-05-16T15:55:28.363Z,2023-05-16T15:55:28.363Z,0,NaN
1,640ef2983f51d833637453f5,jgw-5158254C-33FD-4A66-81D3-1F9FD042AD26-madag...,jgw,madagascar,madagascar -> mauritius,None,In Mauritius: Ministry of Labour,mol@govmu.org,2023-03-13T09:53:28.125Z,2023-03-13T09:53:28.125Z,0,english
2,638add6d292f141036c92d08,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.694Z,2022-12-03T05:23:57.694Z,0,বাংলা
3,638add6d292f141036c92d07,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.494Z,2022-12-03T05:23:57.494Z,0,বাংলা
4,638add6d292f141036c92d06,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.275Z,2022-12-03T05:23:57.275Z,0,বাংলা


In [11]:
# generics
def count_unique(df, field):
    return df[field].nunique()

In [14]:
# phone_email_comparision

def phone_email_comparision(df):
    return {
        "phone": count_unique(df, "phoneNumber"),
        "email": count_unique(df, "email")
    }
    
phone_email_comparision(hc_df)

{'phone': 70, 'email': 11}

In [78]:
# phone numbers and emails by route
def total_phonenumbers_and_emails_clicks_by_org(df):
    df["organization"] = df["organization"].replace("","Not specified")
    
    phonenumber_and_emails_by_org_df = df.groupby("organization")[["phoneNumber", "email"]].agg({
        "phoneNumber": lambda phoneNumbers: phoneNumbers.count(),
        "email": lambda emails: emails.count()
    })
    return {
        "labels": phonenumber_and_emails_by_org_df.index.values,
        "phoneNumbers": phonenumber_and_emails_by_org_df["phoneNumber"].values,
        "email": phonenumber_and_emails_by_org_df["email"].values
    }

dd = total_phonenumbers_and_emails_clicks_by_org(hc_df)
# type(dd)
# dd


In [81]:
# phone numbers and emails by route
def total_phonenumbers_and_emails_clicks_by_route(df):
    df["route"] = df["route"].replace("","Not specified")
    
    phonenumber_and_emails_by_route_df = df.groupby("route")[["phoneNumber", "email"]].agg({
        "phoneNumber": lambda phoneNumbers: phoneNumbers.count(),
        "email": lambda emails: emails.count()
    })
    return {
        "labels": phonenumber_and_emails_by_route_df.index.values,
        "phoneNumbers": phonenumber_and_emails_by_route_df["phoneNumber"].values,
        "email": phonenumber_and_emails_by_route_df["email"].values
    }

dd = total_phonenumbers_and_emails_clicks_by_route(hc_df)
dd

{'labels': array(['Bangladesh -> uk', 'Bulgaria -> uk', 'Greece -> uk',
        'India -> uk', 'Italy -> uk',
        'Other - The rest of the world -> uk', 'Romania -> uk',
        'TJK -> GBR', 'Ukraine -> uk', 'bangladesh -> malaysia',
        'bangladesh -> mauritius', 'burma -> malaysia',
        'india -> malaysia', 'indonesia -> malaysia', 'kenya -> qatar',
        'kenya -> saudi', 'kenya -> uae', 'madagascar -> mauritius',
        'nepal -> malaysia', 'pakistan -> malaysia', 'uganda -> saudi',
        'vietnam -> malaysia'], dtype=object),
 'phoneNumbers': array([ 2,  3, 35,  1,  1,  1, 12,  1,  2, 59, 14, 32,  5, 35, 10,  3,  1,
         7, 25,  0,  3,  2]),
 'email': array([2, 0, 0, 0, 0, 1, 0, 1, 1, 9, 4, 4, 0, 4, 1, 0, 3, 4, 2, 1, 0, 0])}